In [ ]:
import cv2
import numpy as n
import matplotlib.pyplot as p
import os

def show_image(val, im, c=None):
    if im is None or im.size == 0:
        print(f"{val}: no image")
        return

    p.figure(figsize=(5, 5))
    if len(im.shape) == 2:
        p.imshow(im, cmap=c or "gray")
    else:
        p.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    p.title(val)
    p.axis("off")
    p.show()

def show(val, im , c=None):
    p.figure(figsize=(5,5))
    if len(im.shap)==2:
        p.imshow(im,cmap=c)
    else:
        p.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)),
        p.title(val)
        p.axis("off")
        p.show()



def working(im):
    print(im)
    if im is None or im.size == 0:
        return None, None

    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(n.float32)
    print(rgb,"rgb")
    b=rgb[:, :, 0]
    g=rgb[:, :, 1]
    r=rgb[:, :, 2]
    sum__color = r + g + b
    sum__color[sum__color == 0] = 1.0
    x = r / sum__color
    y = g / sum__color
    return x, y

def model():
    allx = []
    ally = []
    files = sorted([
        f for f in os.listdir("skin dataset")
        if f.lower().endswith((".jpg", ".png", ".jpeg"))
    ])

    for fil_n in files:
        path = os.path.join("skin dataset", fil_n)
        im = cv2.imread(path)
        if im is None:
            continue

        roi = cv2.selectROI("Select skin area", im, showCrosshair=True, fromCenter=False)
        x, y, w, h = map(int, roi)
        print(x,y,w,h,"selecting image")
        if w > 0 and h > 0:
            sample = im[y:y+h, x:x+w]
            print("c",sample,"sample")
            print(f"Selected skin: {fil_n}")
            show_image(f"Selected Skin ROI - {fil_n}", sample)
            x, y = working(sample)
            print(x,y)
            allx.extend(x.flatten())
            ally.extend(y.flatten())
            print(allx,ally)
    cv2.destroyAllWindows()

    data = n.stack((n.array(allx), n.array(ally)), axis=0)
    mu = n.mean(data, axis=1)
    covariance = n.cov(data) + n.eye(2) * 1e-6
    return mu, covariance

def detect(im, mean, cov, threashold=3.0):
    x, y = working(im)
    row, col = x.shape
    pixel = n.stack((x.flatten(), y.flatten()), axis=0)
    di = pixel - mean[:, n.newaxis]
    inv = n.linalg.inv(cov)
    dist = n.sum(di * (inv @ di), axis=0)
    mask = (dist < threashold).reshape(row, col).astype(n.uint8) * 255
    return mask


mean_model, cov_model = model()

test_f = sorted([
    f for f in os.listdir("skin dataset/test")
    if f.lower().endswith((".jpg", ".png", ".jpeg"))
])
q=0
while 23>q :
    test = test_f[q]
    q=q+1
    tesimg = cv2.imread(os.path.join("skin dataset/test", test))

    if tesimg is not None:
        print(f"Test image: {test}")
        value = detect(tesimg, mean_model, cov_model, threashold=3.0)
        result = cv2.bitwise_and(tesimg, tesimg, mask=value)
        p.figure(figsize=(15, 5))
        p.subplot(1, 3, 1)
        p.imshow(cv2.cvtColor(tesimg, cv2.COLOR_BGR2RGB))
        p.title("Original")
        p.axis("off")
        p.subplot(1, 3, 2)
        p.imshow(value, cmap="gray")
        p.title("Skin Mask")
        p.axis("off")
        p.subplot(1, 3, 3)
        p.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
        p.title("Predicted Skin")
        p.axis("off")
        cv2.imwrite(os.path.join("skin dataset/test/mask", test),result)
        p.show()
    else:
        print("run out")